In [20]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler


In [21]:
df = pd.read_csv("data/cars.csv")

# using the 99th percentile removes most outliers
price_upper = df["price"].quantile(0.99)
df_clean = df[df["price"] <= price_upper].copy()

In [22]:
X_train, X_test = train_test_split(
    df_clean,
    test_size=0.30,
    random_state=42,
)

y_train = X_train["price"]
y_test = X_test["price"]

### Ridge regression experiments

Use this notebook for regularized linear models, especially experiments that add one-hot encoded categorical features such as `make` and `model`.

### Ridge with cross validation

Polynomial features: ["age", "odometer"]<br>
One-hot features: ["make", "model"]<br>

This experiment creates degree-2 polynomial numeric features, one-hot encodes categorical features, and evaluates a Ridge model with 5-fold cross-validation before checking test-set performance. Find the results below.

In [23]:
numeric_features = ["age", "odometer"]
categorical_features = ["make", "model"]

numeric_transformer = Pipeline([
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("select_best", SelectKBest(score_func=f_regression, k=3)),
    ("scale", StandardScaler()),
])

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("make", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=1.0))
])

cv_results = cross_validate(
    ridge_pipe,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
)

cv_performance = pd.Series({
    "cv_mae": -cv_results["test_mae"].mean(),
    "cv_rmse": -cv_results["test_rmse"].mean(),
    "cv_r2": cv_results["test_r2"].mean(),
})

ridge_pipe.fit(X_train, y_train)
ridge_preds = ridge_pipe.predict(X_test)

test_performance = pd.Series({
    "test_mae": mean_absolute_error(y_test, ridge_preds),
    "test_rmse": mean_squared_error(y_test, ridge_preds) ** 0.5,
    "test_r2": r2_score(y_test, ridge_preds),
})

prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "ridge_preds": ridge_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(cv_performance)
print(test_performance)
print(prediction_summary)

cv_mae     3789.230631
cv_rmse    5442.083658
cv_r2         0.825516
dtype: float64
test_mae     3773.371809
test_rmse    5432.836139
test_r2         0.826414
dtype: float64
              y_test   ridge_preds
mean    19318.045232  19307.568073
std     13039.824946  11810.499117
median  16591.000000  18763.491840
min      1000.000000 -17052.626054
max     65500.000000  62863.270885


### Ridge with cross validation and log-transformed price

This experiment creates degree-2 polynomial features from `age` and `odometer`, keeps the best 3 numeric polynomial features, scales them, one-hot encodes `make` and `model`, and evaluates a Ridge model with 5-fold cross-validation before checking test-set performance.


This repeats the previous Ridge experiment, but trains on `log1p(price)` and converts predictions back with `expm1` so predictions stay on the original dollar scale.

In [24]:
numeric_features = ["age", "odometer"]
categorical_features = ["make", "model"]

numeric_transformer = Pipeline([
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("select_best", SelectKBest(score_func=f_regression, k=3)),
    ("scale", StandardScaler()),
])

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("make", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=Ridge(alpha=1.0),
        func=np.log1p,
        inverse_func=np.expm1,
    ))
])

cv_results = cross_validate(
    ridge_pipe,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
)

cv_performance = pd.Series({
    "cv_mae": -cv_results["test_mae"].mean(),
    "cv_rmse": -cv_results["test_rmse"].mean(),
    "cv_r2": cv_results["test_r2"].mean(),
})

ridge_pipe.fit(X_train, y_train)
ridge_preds = ridge_pipe.predict(X_test)

test_performance = pd.Series({
    "test_mae": mean_absolute_error(y_test, ridge_preds),
    "test_rmse": mean_squared_error(y_test, ridge_preds) ** 0.5,
    "test_r2": r2_score(y_test, ridge_preds),
})

prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "ridge_preds": ridge_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(cv_performance)
print(test_performance)
print(prediction_summary)

cv_mae     3467.217503
cv_rmse    5501.244025
cv_r2         0.821704
dtype: float64
test_mae     3458.711440
test_rmse    5501.787543
test_r2         0.821980
dtype: float64
              y_test    ridge_preds
mean    19318.045232   18783.557334
std     13039.824946   12827.083717
median  16591.000000   15515.574322
min      1000.000000    1039.250963
max     65500.000000  100224.962135


### Ridge with grid search and log-transformed price

This repeats the log-transformed Ridge experiment, but uses `GridSearchCV` to choose the Ridge regularization strength before evaluating the best model on the test set.

In [25]:
numeric_features = ["age", "odometer"]
categorical_features = ["make", "model"]

numeric_transformer = Pipeline([
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("select_best", SelectKBest(score_func=f_regression, k=3)),
    ("scale", StandardScaler()),
])

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("make", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=Ridge(alpha=1.0),
        func=np.log1p,
        inverse_func=np.expm1,
    ))
])

param_grid = {
    "model__regressor__alpha": [0.1, 1.0, 10.0, 100.0, 1000.0]
}

grid_search = GridSearchCV(
    ridge_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

grid_search.fit(X_train, y_train)

cv_performance = pd.Series({
    "best_alpha": grid_search.best_params_["model__regressor__alpha"],
    "best_cv_rmse": -grid_search.best_score_,
})

ridge_preds = grid_search.predict(X_test)

test_performance = pd.Series({
    "test_mae": mean_absolute_error(y_test, ridge_preds),
    "test_rmse": mean_squared_error(y_test, ridge_preds) ** 0.5,
    "test_r2": r2_score(y_test, ridge_preds),
})

prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "ridge_preds": ridge_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(cv_performance)
print(test_performance)
print(prediction_summary)

best_alpha         1.000000
best_cv_rmse    5501.244025
dtype: float64
test_mae     3458.711440
test_rmse    5501.787543
test_r2         0.821980
dtype: float64
              y_test    ridge_preds
mean    19318.045232   18783.557334
std     13039.824946   12827.083717
median  16591.000000   15515.574322
min      1000.000000    1039.250963
max     65500.000000  100224.962135


### Ridge with cross validation and log-transformed price

This repeats the previous Ridge experiment, but trains on `log1p(price)` and converts predictions back with `expm1` so predictions stay on the original dollar scale.

In [26]:
numeric_features = ["age", "odometer"]
categorical_features = ["make", "model", "fuel", "transmission"]

numeric_transformer = Pipeline([
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("select_best", SelectKBest(score_func=f_regression, k=3)),
    ("scale", StandardScaler()),
])

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("make", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=Ridge(alpha=1.0),
        func=np.log1p,
        inverse_func=np.expm1,
    ))
])

cv_results = cross_validate(
    ridge_pipe,
    X_train,
    y_train,
    cv=5,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
)

cv_performance = pd.Series({
    "cv_mae": -cv_results["test_mae"].mean(),
    "cv_rmse": -cv_results["test_rmse"].mean(),
    "cv_r2": cv_results["test_r2"].mean(),
})

ridge_pipe.fit(X_train, y_train)
ridge_preds = ridge_pipe.predict(X_test)

test_performance = pd.Series({
    "test_mae": mean_absolute_error(y_test, ridge_preds),
    "test_rmse": mean_squared_error(y_test, ridge_preds) ** 0.5,
    "test_r2": r2_score(y_test, ridge_preds),
})

prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "ridge_preds": ridge_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(cv_performance)
print(test_performance)
print(prediction_summary)

cv_mae     3376.171251
cv_rmse    5345.289244
cv_r2         0.831670
dtype: float64
test_mae     3376.092025
test_rmse    5380.099494
test_r2         0.829768
dtype: float64
              y_test    ridge_preds
mean    19318.045232   18824.832953
std     13039.824946   12964.862715
median  16591.000000   15528.388067
min      1000.000000    1025.094491
max     65500.000000  102880.751291


### Ridge with grid search log-transformed price

This repeats the log-transformed Ridge experiment, but uses `GridSearchCV` to choose the Ridge regularization strength before evaluating the best model on the test set.

In [27]:
numeric_features = ["age", "odometer"]
categorical_features = ["make", "model", "fuel", "transmission"]

numeric_transformer = Pipeline([
    ("polynomial_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("select_best", SelectKBest(score_func=f_regression, k=3)),
    ("scale", StandardScaler()),
])

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("make", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", TransformedTargetRegressor(
        regressor=Ridge(alpha=1.0),
        func=np.log1p,
        inverse_func=np.expm1,
    ))
])

param_grid = {
    "model__regressor__alpha": [0.25, 0.5, 1, 2, 5]
}

grid_search = GridSearchCV(
    ridge_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

grid_search.fit(X_train, y_train)

cv_performance = pd.Series({
    "best_alpha": grid_search.best_params_["model__regressor__alpha"],
    "best_cv_rmse": -grid_search.best_score_,
})

ridge_preds = grid_search.predict(X_test)

test_performance = pd.Series({
    "test_mae": mean_absolute_error(y_test, ridge_preds),
    "test_rmse": mean_squared_error(y_test, ridge_preds) ** 0.5,
    "test_r2": r2_score(y_test, ridge_preds),
})

prediction_summary = pd.DataFrame({
    "y_test": y_test,
    "ridge_preds": ridge_preds,
}).agg(["mean", "std", "median", "min", "max"])

print(cv_performance)
print(test_performance)
print(prediction_summary)

best_alpha         2.000000
best_cv_rmse    5345.262378
dtype: float64
test_mae     3375.776556
test_rmse    5378.162008
test_r2         0.829890
dtype: float64
              y_test    ridge_preds
mean    19318.045232   18819.106462
std     13039.824946   12951.750552
median  16591.000000   15545.400209
min      1000.000000    1024.425621
max     65500.000000  102598.948832
